In [ ]:
# obtain gguf
# read set values from yaml
!set model_url
!set GIT_LFS_SKIP_SMUDGE=1 && git clone $model_url
!set model_name=regex model_url_rfind_slash
!python llama.cpp/convert_hf_to_gguf.py $model_name --outfile ./$model_name+'.gguf'
!llama-quantize $model_name.gguf $model_name+$quantization_type.gguf $quantization_type N=?-1?

In [1]:
import benchmark_llm
import time
for model_name in ["smolm2-pruned-wanda-q80"]:
    model_path=f'/mnt/d/Inno/Thesis/llm_on_edge/benchmark_llm/{model_name}.gguf'
    benchllm=benchmark_llm.BenchLLM('../../llama.cpp/bin/llama-server','../../llama.cpp/bin/llama-perplexity',model_path)
    time.sleep(60) # wait for previous model free and server to close
    benchllm.start_server()
    time.sleep(240) # wait for model to load
    benchllm.bench() # stop server itself
    benchllm.save_to_file(f'./{model_name}')

In [2]:
import benchmark_llm
benchllm=benchmark_llm.BenchLLM('../../llama.cpp/bin/llama-server','../../llama.cpp/bin/llama-perplexity','##')
result_df=benchllm.compare_results('.')
result_df

,method,token/sec,accuracy,ppl,flips
0,smollm2-iq4xsjson.json,31.685873202164807+-4.679285929058822,0.115,11.218033+-0.6112230871147706,0.155
1,smollm2-q16sparsegpt.json,11.017353771035145+-0.2882285739400089,0.145,18.0922205+-1.3367107806046428,0.145
2,smollm2-q16wanda.json,10.607968182282585+-0.8920774742859577,0.20212765957446807,16.001023+-0.8810941569081527,nan
3,smollm2-q4km.json,29.627005844926085+-3.470098915617861,0.145,10.983886+-0.6253000362069413,0.145
4,smollm2-q80json.json,18.873430371655708+-2.037660093955812,0.115,10.43017+-0.5873723379740362,0.16
5,smolm2-pruned-sparsegpt-iq4xs.json,26.68510706461849+-1.1742060280773141,0.185,18.1986055+-1.1910230139753153,0.15
6,smolm2-pruned-sparsegpt-q80.json,17.89948699414894+-1.4969481381375933,0.165,17.905448999999997+-1.3215589139779478,0.145
7,smolm2-pruned-wanda-iq4xsjson.json,30.164995655055037+-2.920759112419593,0.18,18.2734045+-1.0561964293713713,0.135
8,smolm2-pruned-wanda-q80.json,19.699137229096365+-1.6676730128032948,0.19,16.040091500000003+-0.8730343117686595,0.14


In [3]:
benchllm.calc_corr(result_df)

,token/sec,accuracy,ppl,flips
token/sec,1.00000000000000000000,-0.23932453330544870007,-0.31598010346616334232,-0.11164670167321617822
accuracy,-0.23932453330544870007,1.00000000000000000000,0.73401708796053677375,-0.72445564995623756843
ppl,-0.31598010346616334232,0.73401708796053677375,1.00000000000000000000,-0.65529464936761339100
flips,-0.11164670167321617822,-0.72445564995623756843,-0.65529464936761339100,1.00000000000000000000


In [ ]:
# non pruned to bf16 gguf